# Lesson 4 — Algorithms and Big-O
### Intro to Computer Science for Non-CS Students

**The bridge from Lessons 1 and 2.** Two threads run through everything so far:

1. **Lesson 1:** a structure is a bundle of properties — ordering, uniqueness, access
   pattern — and *operations reveal properties*. We said a dictionary lookup is
   "designed for lookup" and a list scan "repeats work", but we never said *how much*.
2. **Lesson 2:** pandas gave us verbs — filter, sort, `groupby`, `merge` — and they
   *felt* instant on 20 rows. On 20 million rows, some of them stay instant and some
   become coffee breaks.

This lesson supplies the missing piece: **every operation has a price, and the
properties of a structure set that price.** An *algorithm* is the sequence of
operations behind a question, and *Big-O* is the unit we use to talk about its price —
not in seconds, but in **how the work grows when the data grows**.

```
Business question               "Which customer spent the most?"
        ↓
Data structure     (Lesson 1)   list? dictionary? DataFrame?
        ↓
Algorithm          (this one)   scan · sort · look up · match
        ↓
Efficiency         (this one)   how does the work grow with the data?
        ↓
Implementation     (Lesson 2)   Python loop · dict · pandas
```

**Prerequisites and setup**

- Lesson 1: lists, dictionaries, sets, and the property checklist.
- Lesson 2: boolean filtering, `sort_values`, `groupby`, `merge`.
- Uses `pandas` and `matplotlib`, the same course environment as earlier lessons.
- Run top to bottom — later cells depend on earlier ones.

**Suggested sessions:** ① Unit 1 · ② Unit 2 · ③ Unit 3 · ④ Unit 4 + wrap-up

**Practice** lives in the companion notebook `Lesson3_Exercises.ipynb` —
auto-graded, like Lesson 1's.

In [ ]:
import bisect
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

print("pandas", pd.__version__)
print("Setup complete — no files needed this lesson.")

## Unit 1 — What is an algorithm?  ·  ~75 min

Strip away the intimidating name and an algorithm is just a **repeatable procedure
that turns an input into an output**:

- a recipe (ingredients → steps → dish)
- a month-end close checklist (raw ledgers → steps → financial statement)
- long division (two numbers → steps → quotient)

You have been *executing* algorithms since Lesson 1. Today we make them visible,
because once you can see the procedure, you can ask the two questions this whole
lesson is about: *what is the computer actually doing?* and *how much of it?*

### The opening problem

Here are five months of sales. Answer the four questions **by eye** first — really
do it — and pay attention to what your eyes are doing.

1. What was the highest sales value?
2. What were total sales?
3. How many months were above 1,200?
4. *Which month* (position) had the highest sales?

In [ ]:
sales = [1200, 950, 1400, 1800, 1250]
sales

You almost certainly moved left to right, kept a running "best so far" or a running
total in your head, and gave an answer at the end. **That procedure is the
algorithm.** A computer needs the same procedure — just spelled out, because it has
no eyes and no intuition, only instructions.

### Algorithm anatomy — three parts, always

```
Input:      what data do we have?          sales = [1200, 950, 1400, 1800, 1250]
Procedure:  what steps transform it?       examine each value, keep a running total
Output:     what answer do we want?        one number: total sales
```

We will reuse this template constantly. It is the same *input → steps → output*
shape as Lesson 2's workflow diagram — a workflow is just an algorithm whose steps
are themselves big operations.

In [ ]:
# Input:      sales (a list — Lesson 1: ordered, position-based, duplicates allowed)
# Procedure:  start at 0; examine each value once; add it to the running total
# Output:     one number

total = 0
for value in sales:
    total += value

print("Total sales:", total)

### The linear scan — the workhorse of analytics

Most day-to-day analytics questions are answered by one pattern: **examine every
observation exactly once, carrying a little bit of state along the way.** That
pattern is called a *linear scan*. Finding the maximum makes the carried state
visible — the "best seen so far":

In [ ]:
largest = sales[0]          # the first value starts as the best seen so far

for value in sales:
    if value > largest:     # a better value appears...
        largest = value     # ...so it becomes the new best

print("Highest month:", largest)

Walk the trace by hand once: `1200` starts as best → `950` loses → `1400` wins →
`1800` wins → `1250` loses → answer `1800`. Five values, five examinations, no
value looked at twice.

And of course, real Python code says:

In [ ]:
print("Highest month:", max(sales))

> **The teaching point of the whole unit:** `max()` did not *eliminate* the
> algorithm — Python simply ran the scan *for you*, in fast compiled code. The
> procedure still happened; every value was still examined.

This distinction — **the algorithm** versus **its implementation** — matters
everywhere. `df["amount"].sum()` *is* the running-total scan. Convenience hides
the procedure; it never removes the cost.

### The four analytics verbs, as scans

One small dataset, four classic operations. Every one of them is a linear scan
with different carried state.

In [ ]:
transactions = [25, 50, 10, 80, 35]

# 1. SUM — carry a running total
total = sum(transactions)

# 2. AVERAGE — a sum scan plus one division
average = sum(transactions) / len(transactions)

print("total:", total, "| average:", average)

In [ ]:
# 3. COUNT values meeting a condition — carry a counter
large_count = 0
for amount in transactions:
    if amount >= 50:
        large_count += 1

print("transactions >= 50:", large_count)

In [ ]:
# 4. FILTER — carry a growing list of the values that pass the test
large_transactions = []
for amount in transactions:
    if amount >= 50:
        large_transactions.append(amount)

print("kept:", large_transactions)

# The same filter, in Python's compact "comprehension" form:
large_transactions = [amount for amount in transactions if amount >= 50]
print("kept (comprehension):", large_transactions)

### You already know these algorithms — you met them wearing pandas costumes

| Plain-Python scan (today) | Lesson 2 verb | Carried state |
| --- | --- | --- |
| running total | `df["amount"].sum()` | the total so far |
| best-so-far | `df["amount"].max()` | the best so far |
| condition counter | `(df["amount"] >= 50).sum()` | the count so far |
| keep-if-passes | `df[df["amount"] >= 50]` | the survivors so far |

Same input → steps → output. pandas just runs the steps in compiled code, on whole
columns at once (Lesson 1's *homogeneity* property is what makes that possible).

> **Unit 1 takeaway:** most basic data analysis is repeated scanning, comparing,
> counting, filtering, and combining. When you meet a new tool, ask: *which scan is
> this running for me?*

**Try it before Session 2:** for
`unemployment_rates = [4.2, 4.0, 3.8, 4.1, 4.5, 4.3]`, describe in plain English —
*then* write — the scans for: highest rate, average rate, count above 4%, and the
list of rates below 4%.

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. Every algorithm has three parts. Name them.
2. Describe the procedure behind `max(sales)` in plain English.
3. Does calling `max()` mean no algorithm ran? What actually happened?
4. Counting values above 50 and filtering values above 50 share the same basic
   move. What is it, and what differs?
5. From Lesson 2: `df[df["amount"] >= 50]` is which of today's scans in disguise?

<details><summary><b>Answers</b></summary>

1. Input, procedure (the steps), output.
2. Take the first value as the best so far; examine each value once; replace the
   best whenever a larger value appears; report the final best.
3. The scan still ran — Python executed it for us in compiled code. Convenience
   hides the procedure, never the cost.
4. Both examine every value once and test a condition. The *carried state* differs:
   a counter versus a growing list of survivors.
5. The filter scan — keep each row that passes the test.
</details>

## Unit 2 — Searching and sorting: algorithms in action  ·  ~80 min

Unit 1 gave us the idea of a procedure. Now we will build and run three concrete algorithms before giving their growth patterns formal names. One question drives
everything here:

> **"Find me customer 309."** — how, and at what cost?

Three answers, three prices: scan the list (O(n)), binary-search a *sorted* list
(O(log n)), or look up a dictionary key (O(1) on average). The punchline, which
Lesson 1 set up weeks in advance: **the properties you give a structure determine
which algorithms it unlocks.**

### 2.1 Linear search — the honest default

In [ ]:
customer_ids = [105, 207, 114, 309, 411]
target = 309

found = False
for customer_id in customer_ids:
    if customer_id == target:
        found = True
        break                 # stop early — no reason to keep scanning

print("found:", found)

The `break` helps when we get lucky, but to compare algorithms fairly we ask about the **worst case**: the
target might be the last item — or *absent*, which forces a full scan just to prove
it. So a linear search may inspect all n values. (By the way, `target in customer_ids` runs exactly
this scan for you. Convenience hides the procedure…)

One search over one list: perfectly fine. The pain arrives with *repeated*
searches — 1,000 lookups against a 1,000,000-row list is a billion comparisons.
Hold that thought.

### 2.2 Binary search — sorting unlocks the halving move

Now suppose the list is **sorted**:

```
[105, 114, 207, 309, 411]      looking for 309
```

Check the **middle** value (207). Too small — and *because the list is sorted*,
every value to the left of 207 must also be too small. One comparison just
eliminated half the list:

```
[105, 114, 207, 309, 411]
           ↑ too small — discard the entire left half
              [309, 411]
                ↑ found it
```

Each check halves what remains: a million rows need only about 20 checks; a billion need about 30. We will name this growth pattern in Unit 3. Watch the halving happen on a bigger list:

In [ ]:
sorted_ids = list(range(100, 100 + 2 * 5000, 2))    # 5,000 sorted even-numbered IDs
target = 8200

low, high = 0, len(sorted_ids) - 1
checks = 0
while low <= high:
    middle = (low + high) // 2
    checks += 1
    remaining = high - low + 1
    print(f"check {checks:>2}: {remaining:>5} candidates remain, middle value = {sorted_ids[middle]}")
    if sorted_ids[middle] == target:
        print(f"-> found {target} after {checks} checks "
              f"(a linear scan would have needed {sorted_ids.index(target) + 1:,})")
        break
    elif sorted_ids[middle] < target:
        low = middle + 1      # discard the left half
    else:
        high = middle - 1     # discard the right half

Five thousand candidates dissolve in about a dozen checks — watch the `remaining`
column halve on every line. In practice you don't write this loop yourself;
Python's `bisect` module carries it:

In [ ]:
position = bisect.bisect_left(sorted_ids, target)
if position < len(sorted_ids) and sorted_ids[position] == target:
    print("Found at position", position)
else:
    print("Not found")

No need to memorize `bisect`. The concept is the exam material:

> **Binary search requires sorted data.** The halving move is only *legal* because
> order guarantees "everything left of a too-small value is also too small". Run
> it on an unsorted list and it confidently returns wrong answers — no error, just
> nonsense.

In Lesson 1's language: **ordering is a property, and this property unlocks an
algorithm.** An unsorted list simply does not offer the O(log n) search, the same
way a set does not offer "the third item".

### 2.3 Sorting — how data acquires the ordering property

Basics first:

In [ ]:
revenues = [25000, 18000, 31000, 22000]

print("ascending: ", sorted(revenues))
print("descending:", sorted(revenues, reverse=True))

Real datasets are records, not bare numbers — so we tell `sorted()` **which field
to compare**, using `key=`:

In [ ]:
companies = [
    {"name": "Company A", "revenue": 25000},
    {"name": "Company B", "revenue": 18000},
    {"name": "Company C", "revenue": 31000},
]

ranked = sorted(companies, key=lambda company: company["revenue"], reverse=True)

for position, company in enumerate(ranked, start=1):
    print(f"{position}. {company['name']}: {company['revenue']:,}")

(`lambda` is a one-line unnamed function: *given a company, return its revenue*.
The `key` tells `sorted()` what number to compare records by.)

This is the algorithm behind every leaderboard you will ever build — top customers
by lifetime value, countries by GDP growth, products by margin — and it is exactly
Lesson 2's `df.sort_values("revenue", ascending=False)`, at O(n log n).

### 2.4 The preparation trade-off — when is sorting worth it?

Sorting costs O(n log n). Why ever pay it?

| Strategy | Cost for q searches over n rows |
| --- | --- |
| Just scan, every time | Up to q × n checks |
| Sort once, then binary search each time | One sort + about q × log₂(n) checks |

No algebra needed — the business logic is enough:

- **One search?** Just scan. Sorting first is building a filing cabinet to find
  one document.
- **Thousands of searches?** Sort (or index) first. The preparation cost is repaid
  on every single lookup.

> **Sometimes we spend time organizing data so that future questions become cheap.**

You have met this idea before without the name: Lesson 2's `set_index` and `.loc`
label lookups. And it is exactly why databases build **indexes** — a prepared,
ordered companion structure that turns full-table scans into O(log n) jumps.
Preparation paid once; savings collected forever.

### 2.5 The dictionary — skip the search entirely

Binary search made lookup cheap. The dictionary makes it *almost free*. Recall the
Lesson 1 pair:

In [ ]:
# Representation 1: a list of records -> finding a customer means scanning: O(n)
customer_records = [
    {"id": 101, "name": "Alice"},
    {"id": 102, "name": "Bob"},
    {"id": 103, "name": "Carol"},
]

# Representation 2: a dictionary keyed by ID -> O(1) on average
customers_by_id = {
    101: "Alice",
    102: "Bob",
    103: "Carol",
}

print(customers_by_id[102])       # no scan happened here

How can it *not* scan? A dictionary is Lesson 1's **hash map**: it converts the
key into a storage location and jumps straight there — the same trick that makes
`sales[0]` instant, extended from positions to arbitrary keys. That is why we say
*average* O(1): rare unlucky arrangements degrade it, but in practice you may
treat dictionary lookup as constant-time.

Time to see the price difference. Same data, same 1,000 membership questions — the
only change is the structure (a set uses the same hash trick as dict keys):

In [ ]:
n = 100_000
id_list = list(range(n))          # 100,000 IDs in a list
id_set = set(id_list)             # the same IDs, hash-organized

random.seed(42)
lookups = [random.randrange(n * 2) for _ in range(1_000)]   # ~half will be misses

start = time.perf_counter()
hits_list = sum(1 for target in lookups if target in id_list)   # O(n) scan per lookup
list_seconds = time.perf_counter() - start

start = time.perf_counter()
hits_set = sum(1 for target in lookups if target in id_set)     # O(1) avg per lookup
set_seconds = time.perf_counter() - start

assert hits_list == hits_set
print("1,000 membership checks against 100,000 IDs")
print(f"  list (scan each time): {list_seconds:8.4f} s")
print(f"  set  (hash lookup):    {set_seconds:8.6f} s")
print(f"  speedup: ~{list_seconds / set_seconds:,.0f}x")

Identical data, identical question, **thousands of times faster** — because
changing the structure's access-pattern property changed the algorithm from scan
to jump.

> **Unit 2 takeaway:** search has three prices — O(n) scan, O(log n) after
> sorting, O(1) average by key — and you choose the price when you choose the
> structure. Lesson 1 told you *what* each structure is for; today you learned
> *how much* that choice is worth.

---
## ⏱️ Session 3 warm-up  ·  5 questions from last time

1. Why does binary search require sorted data — what exactly goes wrong without it?
2. About how many checks does binary search need in 1,000,000 sorted IDs?
3. When is it worth sorting before searching, and when is it not?
4. Why is a dictionary lookup usually faster than scanning a list?
5. Lesson 1 called *access pattern* a property. What price tags did this lesson
   attach to it?

<details><summary><b>Answers</b></summary>

1. The halving step assumes "middle too small → the whole left half is too small."
   Unsorted data breaks that assumption, so halves containing the target get
   discarded — and the search confidently answers wrong.
2. About 20 (each check halves what remains; 2²⁰ ≈ 1,000,000).
3. Many repeated searches → sort or index once, collect O(log n) on every lookup.
   A single search → just scan; preparation would cost more than it saves.
4. It hashes the key into a storage location and jumps there — no scanning: O(1)
   average versus O(n).
5. Access by position O(1) · linear scan O(n) · binary search on sorted data
   O(log n) · dictionary key lookup O(1) average.
</details>

## Unit 3 — Big-O: naming how real algorithms grow  ·  ~80 min

You have now run scans, binary search, sorting, and hash lookups. On five values they all feel instant. Their differences appear when the data grows:

> Not *"how fast is this code today?"* but **"how does the amount of work grow as
> the dataset grows?"**

Suppose one basic operation takes about a microsecond (a millionth of a second):

| Rows | Examine each row once | Compare every pair of rows |
| ---: | ---: | ---: |
| 10 | 10 ops — instant | 100 ops — instant |
| 1,000 | 1,000 ops — instant | 1,000,000 ops — ~1 second |
| 1,000,000 | 1,000,000 ops — ~1 second | 1,000,000,000,000 ops — **~11 days** |

Both columns look identical on a small test file. Only one of them survives contact
with real data. **Big-O notation** is the vocabulary for the difference: we write
the growth pattern as O(*something*), where **n is the size of the input** — the
number of rows, customers, transactions, observations.

### The five complexity classes you actually need

There are exactly five patterns worth memorizing for analytics work. From cheapest
to most expensive:

---

### O(1) — constant: the work doesn't grow at all

Whether the list holds 3 values or 3 million, grabbing the first one is one step:

In [ ]:
first_sale = sales[0]      # one step on 5 rows, one step on 5 million rows
first_sale

Why can a list do that? Lesson 1's *access pattern* property: a list knows where
position 0 lives, so it jumps straight there — no scanning.

Everyday O(1) operations: access a list item by position · look up a dictionary
value by key (on average — as you saw in Unit 2) · read one known cell of a table.

---

### O(log n) — logarithmic: each step throws away half the problem

Some algorithms discard *half* of the remaining data at every step. The famous one
is **binary search** (from Unit 2). The growth is astonishingly slow:

```
        1,000 items  →  ~10 checks
    1,000,000 items  →  ~20 checks
1,000,000,000 items  →  ~30 checks
```

No logarithm calculations needed. The intuition to keep: **doubling the data adds
just one more step.**

---

### O(n) — linear: examine each item once

Every Unit 1 scan lives here: sum, average, max, min, count, filter, and linear
search. Twice the data → twice the work. Honest, predictable, usually fine.

In [ ]:
def scan_count(data, threshold):
    count = 0
    for value in data:       # n values -> n examinations -> O(n)
        if value > threshold:
            count += 1
    return count

scan_count(sales, 1200)

---

### O(n log n) — the price of efficient sorting

Sorting has to *arrange* every value, which costs more than looking at each one
once — but far, far less than comparing every value with every other value.
Python's built-in `sorted()` is in this class:

In [ ]:
sorted_sales = sorted(sales)
sorted_sales

The intuition to keep: **sorting costs a bit more than a scan, and it is the most
expensive thing you should do routinely.** (Lesson 2's `sort_values` lives here.)

---

### O(n²) — quadratic: compare many items with many items

The danger class. It almost always enters through an innocent-looking **nested
loop** — a loop inside a loop over the same data:

In [ ]:
customers = ["Alice", "Bob", "Carol", "Dana"]

pairs = 0
for customer_a in customers:
    for customer_b in customers:     # a full inner loop for EVERY outer customer
        pairs += 1

print(len(customers), "customers ->", pairs, "comparisons")

4 customers → 16 comparisons. 1,000 customers → 1,000,000. 1,000,000 customers →
a trillion. This is the "compare every pair" column from the opening table: pairwise
similarity, duplicate hunting done naively, and — the big one for Unit 4 — matching
two datasets with nested loops.

O(n²) is not forbidden. On 200 rows it's harmless. The skill is **recognizing** it,
so that when the data is large you reach for a better structure instead.

### Reading complexity straight from code shape

You can usually classify code without deep analysis — the loops give it away.

```python
for value in data:            # one loop over the data
    process(value)                    # -> O(n)
```

```python
for value in data:            # two loops, one AFTER the other
    step_one(value)
for value in data:
    step_two(value)                   # -> O(n + n) = O(2n) = O(n)
```

```python
for first in data:            # a loop INSIDE a loop
    for second in data:
        compare(first, second)        # -> O(n × n) = O(n²)
```

Two things to notice:

- **Sequential loops add; nested loops multiply.** That's the whole trick.
- Big-O **ignores the constant**: O(2n) is written O(n), because both double when
  the data doubles. The *shape* of growth is the same, and only the shape survives
  at scale.

### Measure the growth shapes on this computer

Big-O is about a pattern, so one stopwatch result is not enough. We run the same
real algorithms at several input sizes, repeat each measurement, and plot the
median time. Your exact milliseconds will differ; the **curve shape** is the result.


In [ ]:
def median_runtime(function, *args, repeats=5):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(*args)
        samples.append(time.perf_counter() - start)
    return sorted(samples)[len(samples) // 2]

def linear_scan(data):
    total = 0
    for value in data:
        total += value
    return total

def sort_data(data):
    return sorted(data)

def compare_every_pair(data):
    count = 0
    for first in data:
        for second in data:
            count += first <= second
    return count

sizes = [200, 400, 800, 1_600, 3_200]
scan_times, sort_times, pair_times = [], [], []

for n in sizes:
    rng = random.Random(42)
    data = list(range(n))
    rng.shuffle(data)  # avoids giving sorted() an already ordered best case
    scan_times.append(median_runtime(linear_scan, data))
    sort_times.append(median_runtime(sort_data, data))
    pair_times.append(median_runtime(compare_every_pair, data, repeats=3))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sizes, scan_times, "o-", linewidth=2, label="linear scan")
ax.plot(sizes, sort_times, "o-", linewidth=2, label="sorting")
ax.plot(sizes, pair_times, "o-", linewidth=2, label="compare every pair")
ax.set(xlabel="input size n", ylabel="measured time (seconds)",
       title="Real runtime as the input grows")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.grid(True, which="both", alpha=.25)
ax.legend()
plt.show()


Read the plot from left to right. When n doubles, the scan rises by roughly 2×,
sorting rises a little more than 2×, and comparing every pair rises toward 4×.
Small timing noise is normal—the operating system and Python add overhead—but the
growth shapes separate quickly.


In [ ]:
# Make the doubling pattern visible as measured ratios.
print(f"{'n':>7} | {'scan ×':>8} | {'sort ×':>8} | {'pairs ×':>8}")
print("-" * 41)
for i in range(1, len(sizes)):
    print(f"{sizes[i]:>7,} | "
          f"{scan_times[i] / scan_times[i-1]:>8.2f} | "
          f"{sort_times[i] / sort_times[i-1]:>8.2f} | "
          f"{pair_times[i] / pair_times[i-1]:>8.2f}")


The measured ratios will not be perfect textbook numbers, especially for tiny
inputs. That is useful evidence, not a failure: **Big-O is not a stopwatch.** It
does not know your computer, Python's overhead, or what else the operating system
is doing. It describes the long-run growth pattern.

> **The analogy to keep:** Big-O describes the *shape of the road*, not your travel
> time. Hardware changes the vehicle; growth determines how steep the road becomes.


### Classify these — cover the right column first

| Operation | Complexity |
| --- | --- |
| Read the first transaction in a list | O(1) |
| Calculate total spending | O(n) |
| Find a customer by scanning a list | O(n) |
| Sort customers by revenue | O(n log n) |
| Compare every customer with every other customer | O(n²) |
| Binary search in a sorted customer list | O(log n) |

*(One sentence of "why" beats the right label with no reason — that is also the
grading standard in the exercises notebook.)*

> **Unit 3 takeaway:** n is the size of the data; Big-O is the shape of the work's
> growth. Scans are O(n), sorting is O(n log n), nested loops are O(n²) — and the
> gap between those shapes is the difference between one second and eleven days.

---
## ⏱️ Session 4 warm-up  ·  5 questions from last time

1. What does n stand for in O(n)?
2. Order from slowest-growing to fastest-growing: O(n²), O(1), O(n log n), O(n), O(log n).
3. Two separate (not nested) loops over the same list — what class, and why?
4. Why does Big-O write O(2n) as plain O(n)?
5. A report takes 2 seconds on 10,000 rows. Estimate its time on 1,000,000 rows
   if the code is O(n) — and if it is O(n²).

<details><summary><b>Answers</b></summary>

1. The size of the input — number of rows, customers, transactions.
2. O(1) → O(log n) → O(n) → O(n log n) → O(n²).
3. O(n): sequential loops add (n + n = 2n), and constants are dropped.
4. Both 2n and n *double when the data doubles* — same growth shape. Big-O keeps
   only the shape, because the shape is what survives at scale.
5. O(n): 100× the data → about 200 seconds (~3 minutes). O(n²): 100× the data →
   10,000× the work → about 20,000 seconds — more than five hours.
</details>

## Unit 4 — Aggregation, matching, and joins  ·  ~80 min

The career unit. Lesson 2's two heavyweight verbs were `groupby` and `merge` — you
used them, they worked, and they stayed black boxes. Today we build both **by
hand** out of Unit 1's scans and Unit 2's dictionaries, price them with Unit 3's
Big-O, and then reopen the pandas box to find our own algorithm inside.

### 4.1 Aggregation — totals by category

The business question: *total spending per category.*

In [ ]:
spending = [
    {"category": "Food",   "amount": 25},
    {"category": "Travel", "amount": 80},
    {"category": "Food",   "amount": 15},
    {"category": "Books",  "amount": 30},
    {"category": "Travel", "amount": 50},
]

category_totals = {}                          # category -> running total
for transaction in spending:
    category = transaction["category"]
    amount = transaction["amount"]
    if category not in category_totals:       # first time we meet this category
        category_totals[category] = 0
    category_totals[category] += amount

category_totals

Look at what this is made of: a **linear scan** (Unit 1) whose carried state is a
**dictionary** (Unit 2) holding one running total per category. Each transaction
is examined once; each dictionary update is O(1) average — so the whole
aggregation is **O(n)**. Grouping is not expensive. It is a single pass with a
smart container.

Now reopen Lesson 2's black box:

In [ ]:
spending_df = pd.DataFrame(spending)
spending_df.groupby("category")["amount"].sum()

Same numbers. `groupby` **is** the dictionary-of-running-totals scan —
implemented in compiled code and generalized to any aggregation (`mean`, `count`,
`max`, …), but the same algorithm. Lesson 2's three words *split, apply, combine*
translate to: route each row to its key's bucket, update the bucket, present the
buckets.

### 4.2 Counting frequencies — the same move, carrying counts

In [ ]:
categories = ["Food", "Travel", "Food", "Books", "Travel"]

counts = {}
for category in categories:
    counts[category] = counts.get(category, 0) + 1    # .get: "current count, or 0 if new"

print(counts)

# The standard library ships this exact algorithm ready-made:
print(Counter(categories))

(And Lesson 2's `value_counts()` — same algorithm again.) Libraries hand you the
implementation; understanding the procedure is what lets you predict its cost and
trust its output.

### 4.3 The matching problem — the algorithm inside `merge`

Two datasets, one shared key — Lesson 2's merge setup, rebuilt from scratch:

In [ ]:
customer_table = [
    {"customer_id": 1, "name": "Alice"},
    {"customer_id": 2, "name": "Bob"},
    {"customer_id": 3, "name": "Carol"},
]

order_table = [
    {"customer_id": 2, "amount": 50},
    {"customer_id": 1, "amount": 30},
    {"customer_id": 2, "amount": 20},
]

# Business question: attach the customer NAME to every order.

### 4.4 First attempt — nested loops (spot the shape!)

The instinctive solution: for each order, hunt through the customers.

In [ ]:
matched_orders = []
for order in order_table:                     # n orders...
    for customer in customer_table:           # ...times m customers
        if order["customer_id"] == customer["customer_id"]:
            matched_orders.append({
                "name": customer["name"],
                "amount": order["amount"],
            })

matched_orders

Correct — and you should have recognized the shape from Unit 3 before reading this
sentence: a loop inside a loop. With n orders and m customers this is **O(n × m)**;
when both grow together, effectively **O(n²)**. On 3×3 records, invisible. On
100,000 orders × 50,000 customers: five **billion** comparisons — the classic "my
script worked on the sample and hangs on the real file."

### 4.5 Second attempt — build a lookup, then one scan

Unit 2's medicine: stop *searching* for each customer and *look them up* instead.

- **Step 1 — organize:** one O(m) pass turns customers into a dictionary.
- **Step 2 — scan:** one O(n) pass over orders; each lookup is O(1) average.

In [ ]:
# Step 1: preparation — customers by ID (one pass, O(m))
customer_lookup = {}
for customer in customer_table:
    customer_lookup[customer["customer_id"]] = customer["name"]

# Step 2: one scan over orders (O(n)); each name retrieved in O(1) average
matched_orders = []
for order in order_table:
    matched_orders.append({
        "name": customer_lookup[order["customer_id"]],
        "amount": order["amount"],
    })

matched_orders

Same output, but the price fell from O(n × m) to **O(n + m)** — the two passes
*add* instead of *multiplying*, exactly Unit 3's sequential-versus-nested rule.
Notice it is also Unit 2's preparation trade-off again: pay a small organizing
cost up front, collect the savings on every row.

Don't take the theory's word for it:

In [ ]:
size = 3_000
big_customers = [{"customer_id": i, "name": f"Customer {i}"} for i in range(size)]
big_orders = [{"customer_id": random.randrange(size), "amount": 10} for _ in range(size)]

start = time.perf_counter()
matched_nested = [
    (customer["name"], order["amount"])
    for order in big_orders
    for customer in big_customers
    if customer["customer_id"] == order["customer_id"]
]
nested_seconds = time.perf_counter() - start

start = time.perf_counter()
lookup = {customer["customer_id"]: customer["name"] for customer in big_customers}
matched_dict = [(lookup[order["customer_id"]], order["amount"]) for order in big_orders]
dict_seconds = time.perf_counter() - start

assert matched_nested == matched_dict
print(f"matching {size:,} orders to {size:,} customers")
print(f"  nested loops O(n x m): {nested_seconds:8.3f} s")
print(f"  dictionary   O(n + m): {dict_seconds:8.5f} s")
print(f"  speedup: ~{nested_seconds / dict_seconds:,.0f}x")

Three thousand rows each — small by real-data standards — and the nested version
is already hundreds of times slower. At a million rows it would simply never
finish.

### 4.6 Reopening Lesson 2's `merge`

In [ ]:
customers_df = pd.DataFrame(customer_table)
orders_df = pd.DataFrame(order_table)

orders_df.merge(customers_df, on="customer_id", how="left")

> **A join is not magic. It is an algorithm for matching records on a shared key**
> — and internally pandas uses hash-based lookups: the same build-a-dictionary
> strategy you just wrote, in compiled code.

Lesson 2's merge surprises now have algorithmic explanations too:

- *the inner join dropped `T006`* → its key was simply **absent from the lookup**,
  and inner's rule is "no match, no row" (`how="left"` keeps the row, with blanks).
- *duplicate keys multiplied rows 2×2 → 4* → the lookup held **two** entries for
  that key and each of the 2 rows matched both. The matching algorithm did exactly
  what it was told; the surprise was in the data's uniqueness property, not the code.

> **Unit 4 takeaway:** aggregation = scan + dictionary of running state, O(n).
> Matching = build a lookup + scan, O(n + m) — never nested loops on real data.
> `groupby` and `merge` run these same algorithms for you, fast; now you can
> predict their cost and debug their surprises.

### 4.7 Real-data application — retail transactions

The examples above are deliberately tiny so you can trace each operation. Now use the
same algorithms on a pinned extract of **real** UCI Online Retail transactions. The file
is only 60 rows, so we can read its result in class, but it has the same shape as a real
analysis: transaction rows, customer IDs, prices, quantities, aggregation, sorting, and
matching.

The extract is local and versioned in `../course_data/`, so this demonstration does not need
a network connection.

In [ ]:
from pathlib import Path

retail = pd.read_csv(Path("../course_data") / "lesson2_transactions_base.csv")
customers_real = pd.read_csv(Path("../course_data") / "lesson2_customers_base.csv")
retail["revenue"] = retail["quantity"] * retail["unit_price"]

# Aggregation is one scan, carrying one running total per customer.
revenue_by_customer = {}
for row in retail.to_dict("records"):
    customer_id = row["customer_id"]
    revenue_by_customer[customer_id] = revenue_by_customer.get(customer_id, 0) + row["revenue"]

print("real rows:", len(retail), "| real customers:", retail["customer_id"].nunique())
print("total revenue from one scan:", round(sum(revenue_by_customer.values()), 2))
print()

# Sorting asks the same question as a leaderboard: which real transaction is largest?
top_lines = retail.sort_values("revenue", ascending=False).head(5)
print(top_lines[["transaction_id", "customer_id", "product", "revenue"]])
print()

# Matching the real transaction rows to their customer records.
orders = retail.head(20).to_dict("records")
customer_rows = customers_real.to_dict("records")
nested_matches = [
    (order["transaction_id"], customer["country"])
    for order in orders
    for customer in customer_rows
    if customer["customer_id"] == order["customer_id"]
]
customer_lookup = {customer["customer_id"]: customer["country"] for customer in customer_rows}
lookup_matches = [
    (order["transaction_id"], customer_lookup[order["customer_id"]])
    for order in orders
]

assert nested_matches == lookup_matches
print("nested-loop and dictionary-lookup matches agree:", len(lookup_matches))
print(retail.head(20).merge(customers_real, on="customer_id", how="left")[
    ["transaction_id", "customer_id", "country", "revenue"]
].head())

## Wrap-up

### Six misconceptions this lesson should have broken

| Misconception | Reality |
| --- | --- |
| "Built-in functions have no cost" | `max()` / `sum()` still run an O(n) scan — convenience hides the procedure, not the price |
| "Big-O predicts seconds" | it predicts the *shape of growth*; hardware, language, and constants set the seconds |
| "A faster computer fixes slow code" | 1,000× less data still couldn't save our O(n²) loop — shape beats hardware |
| "Sorting before searching is always smart" | one lookup: just scan; preparation only pays off across many lookups |
| "Dictionaries are always better than lists" | dicts buy O(1) *lookup*; ordered scans, positions, and duplicates are what lists are for |
| "`groupby` and `merge` are magic" | both are this lesson's dictionary algorithms in compiled code — same procedure, smaller constant |

### The lesson, one sentence

> *Ask what the computer actually does (the algorithm), ask how that work grows
> (Big-O), and when the price is too high, change the data's properties — sort it
> or key it — to unlock a cheaper algorithm.*

### The cheat sheet

| Operation | Typical cost | Intuition |
| --- | ---: | --- |
| Access a list item by position | O(1) | jump straight to one place |
| Dictionary / set lookup by key | O(1) average | hash the key, jump there |
| Scan, filter, sum, max, count | O(n) | examine every row once |
| Linear search | O(n) | worst case: it's the last item, or absent |
| Binary search (sorted data only) | O(log n) | discard half per check |
| Sort (`sorted`, `sort_values`) | O(n log n) | the most expensive routine operation |
| Aggregate by key (dict / `groupby`) | O(n) | one scan carrying running totals |
| Match two tables (dict / `merge`) | O(n + m) | build a lookup, then one scan |
| Nested loops over the same data | O(n²) | every pair — the danger class |

### Where each Lesson 1 property got its price tag

- *access pattern* → list position O(1) · scan O(n) · dict key O(1) average — the
  property was always about cost; now the cost has a name
- *ordering* → sorted data unlocks binary search: O(n) becomes O(log n)
- *uniqueness (sets, dict keys)* → membership tests at O(1) average instead of O(n)
- *homogeneity* → why pandas' O(n) beats a Python loop's O(n): same shape, far
  smaller constant
- *"storage is not neutral"* → neither is organization: an index or lookup is
  preparation paid once and collected on every future question

### Looking ahead

When an API someday hands you 100,000 records, these ideas decide whether you
process them with a scan or accidentally write a nested loop — and they are why
servers paginate. And one division of labor worth keeping as LLM tools enter your
workflow: **let Python run the exact algorithms — sums, sorts, matches — and let
the model interpret the verified results**, never the reverse.

### Practice

Open **`Lesson3_Exercises.ipynb`**: seven classic interview problems (real
LeetCode numbers), each one a daily analyst task in disguise, each mapped to a
unit of this lesson — auto-graded test cases plus a Big-O question per problem.

### Resources

- *Grokking Algorithms* (Aditya Bhargava) — the friendliest illustrated introduction; chapters 1–2 cover this lesson
- Python docs: [`bisect`](https://docs.python.org/3/library/bisect.html) · [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- pandas user guide: [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) · [Merge, join, concatenate](https://pandas.pydata.org/docs/user_guide/merging.html)